In [ ]:
#Data
import pandas as pd
import numpy as np
import time

#Sklearn
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from scipy.stats import randint
from sklearn.pipeline import Pipeline
#from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

#Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

names = [

    "Random Forest Default",
    "Random Forest Depth 100",
    "Random Forest Depth 50",
    "Random Forest Depth 25",
    "Random Forest 50 Estimators",
    "Random Forest 50 Estimators Depth 100",
    "Random Forest 50 Estimators Depth 50",
    "Random Forest 50 Estimators Depth 25",
    "Random Forest 25 Estimators",
    "Random Forest 25 Estimators Depth 100",
    "Random Forest 25 Estimators Depth 50",
    "Random Forest 25 Estimators Depth 25"

]

import joblib

#Classifiers
classifiers = [
    #Combine
    RandomForestClassifier(random_state=42), #Default 100 estimators, Pure leafs as depth
    RandomForestClassifier(max_depth = 100, random_state=42),
    RandomForestClassifier(max_depth = 50, random_state=42),
    RandomForestClassifier(max_depth = 25,random_state=42),

    #50 Estimators
    RandomForestClassifier(n_estimators = 50, random_state=42),
    RandomForestClassifier(n_estimators = 50, max_depth = 100, random_state=42),
    RandomForestClassifier(n_estimators = 50, max_depth = 50, random_state=42),
    RandomForestClassifier(n_estimators = 50, max_depth = 25, random_state=42),

    #25 Estimators
    RandomForestClassifier(n_estimators = 25, random_state=42),
    RandomForestClassifier(n_estimators = 25, max_depth = 100, random_state=42),
    RandomForestClassifier(n_estimators = 25, max_depth = 50, random_state=42),
    RandomForestClassifier(n_estimators = 25, max_depth = 25, random_state=42)

]

#Pipeline

data = pd.read_csv('text.csv')


#Split data - Train and test
train_data, test = train_test_split(data, test_size=0.2, random_state=42)
#Split - Train and validation
train, val = train_test_split(train_data, test_size=0.2, random_state=42)


#Avoid data leakage
X_train = train.drop(['issue_title','issue_body','issue_label'], axis=1)
y_train = train['issue_label']

X_val = val.drop(['issue_title','issue_body','issue_label'], axis=1)
y_val = val['issue_label']

X_test = test.drop(['issue_title','issue_body','issue_label'], axis=1)
y_test = test['issue_label']

# Define the preprocessor for each type of feature
preprocessor = ColumnTransformer(
    transformers=[
        ('text1', TfidfVectorizer(), 'token_title'),  # Apply TF-IDF
        ('text2', TfidfVectorizer(), 'token_body'),   # Apply TF-IDF
    ],
    remainder='passthrough'  # Keeps any additional columns as is (if there are any)
)

#Numpy arrays for metrics
accuracy_val = np.zeros(len(names))
precision_val = np.zeros(len(names))
recall_val = np.zeros(len(names))
f1_val = np.zeros(len(names))
accuracy = np.zeros(len(names))
precision = np.zeros(len(names))
recall = np.zeros(len(names))
f1 = np.zeros(len(names))

for i, (name, clf) in enumerate(zip(names, classifiers)):
    # Create a pipeline with the preprocessor and the classifier
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    t0 = time.time()

    print(f"Start training... {name}")
    #Train model
    pipeline.fit(X_train, y_train)
    print("Finish training")

    t_delta = time.time() - t0
    print(f"Training time: {t_delta}")

    #Validation
    print("Validation>>>")
    y_pred_val = pipeline.predict(X_val)
    print(y_pred_val)

    #Metrics
    accuracy_val[i] = accuracy_score(y_val, y_pred_val)
    precision_val[i] = precision_score(y_val, y_pred_val, average='weighted')
    recall_val[i] = recall_score(y_val, y_pred_val, average='weighted')
    f1_val[i] = f1_score(y_val, y_pred_val, average='weighted')

    #Testing
    print(f"Testing>>> {name}")
    y_pred_test = pipeline.predict(X_test)
    print(y_pred_test)

    #Metrics
    accuracy[i] = accuracy_score(y_test, y_pred_test)
    precision[i] = precision_score(y_test, y_pred_test, average='weighted')
    recall[i] = recall_score(y_test, y_pred_test, average='weighted')
    f1[i] = f1_score(y_test, y_pred_test, average='weighted')

    #print(f'Accuracy validation: {accuracy_val} <-> Accuracy testing: {accuracy}')
    #print(f'Precision validation: {precision_val} <-> Precision testing: {precision}')
    #print(f'Recall validation: {recall_val} <-> Recall testing: {recall}')
    #print(f'F1 validation: {f1_val} <-> F1 testing: {f1}')

    #Save in a data set
    metrics = pd.DataFrame({})
    metrics['name'] = names
    metrics['accuracy_val'] = accuracy_val
    metrics['accuracy'] = accuracy
    metrics['precision_val'] = precision_val
    metrics['precision'] = precision
    metrics['recall_val'] = recall_val
    metrics['recall'] = recall
    metrics['f1_val'] = f1_val
    metrics['f1'] = f1
    print(metrics.head())
    metrics.to_csv('metrics.csv', index=False)


Start training... Random Forest Default
Finish training
Training time: 87.33898401260376
Validation>>>
['enhancement' 'bug' 'enhancement' ... 'bug' 'bug' 'enhancement']
Testing>>> Random Forest Default
['bug' 'enhancement' 'enhancement' ... 'bug' 'bug' 'bug']
                          name  accuracy_val  accuracy  precision_val  \
0        Random Forest Default      0.935239  0.936373       0.938002   
1      Random Forest Depth 100      0.000000  0.000000       0.000000   
2       Random Forest Depth 50      0.000000  0.000000       0.000000   
3       Random Forest Depth 25      0.000000  0.000000       0.000000   
4  Random Forest 50 Estimators      0.000000  0.000000       0.000000   

   precision  recall_val    recall   f1_val        f1  
0   0.938819    0.935239  0.936373  0.92746  0.928177  
1   0.000000    0.000000  0.000000  0.00000  0.000000  
2   0.000000    0.000000  0.000000  0.00000  0.000000  
3   0.000000    0.000000  0.000000  0.00000  0.000000  
4   0.000000    0.000